# Tomato Ripeness — SAM3 Auto-Label + YOLOv8n-C3TR Fine-Tune

**Pipeline overview:**
1. Use SAM3 (text-prompted segmentation) to auto-label your images — no manual annotation needed
2. Convert SAM3 outputs → YOLO bounding-box format labels
3. Fine-tune `ripeness_best.pt` (your existing model) on the new labels
4. Validate + export `ripeness_sam3.pt`
5. Demonstrate SAHI inference — slices whole-plant images into tiles so small tomatoes are detected reliably

**Why SAM3 fixes the domain shift problem:**
Your original model was trained on close-up crops and struggles with whole-plant images.
SAM3 uses text prompts (`"unripe green tomato"`) to find tomatoes in YOUR images,
generating labels that already match your camera angle and environment.

**Why SAHI fixes the false-positive / missed-detection problem:**
SAHI slices a 1280x960 whole-plant photo into overlapping 320x320 tiles, runs YOLO
on each tile (where tomatoes look like close-ups), then merges the detections back.
Small tomatoes are no longer tiny specks; backgrounds like walls don't fill a tile.

**Run on Kaggle with:**
- Internet: **ON**
- GPU: **ON** (T4)
- Upload your images as a Dataset input
- Upload `ripeness_best.pt` as a Dataset input
- Kaggle Secret: `HF_TOKEN` (HuggingFace token — SAM3 requires gated access)

**SAM3 model access:**
Request access at https://huggingface.co/facebook/sam3 before running this notebook.
Once approved, create a HuggingFace token and add it as a Kaggle secret named `HF_TOKEN`.

In [ ]:
# Cell 1: Install Dependencies
# sahi = Slicing Aided Hyper Inference (fixes small-object detection in wide shots)
!pip install -q ultralytics huggingface_hub sahi

import os
import shutil
import glob
import random
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from ultralytics.models.sam import SAM3SemanticPredictor
from huggingface_hub import hf_hub_download, login
from IPython.display import Image, display

random.seed(42)
np.random.seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Authenticate HuggingFace + Download SAM3
#
# SAM3 is a gated model — you need to request access at:
#   https://huggingface.co/facebook/sam3
# Then create a token at https://huggingface.co/settings/tokens
# and add it as a Kaggle secret named HF_TOKEN.

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)

SAM3_PATH = hf_hub_download(
    repo_id='facebook/sam3',
    filename='sam3.pt',
    local_dir='/kaggle/working',
)
print(f'SAM3 model: {SAM3_PATH}')
print(f'Size: {os.path.getsize(SAM3_PATH) / (1024**2):.1f} MB')

In [ ]:
# Cell 3: Load Base Model + Point to Your Images
#
# Upload your images as a Kaggle dataset named 'my-tomato-images'.
# Any mix of whole-plant shots, close-ups, indoor/outdoor all work —
# SAM3 handles variety since it reasons from text descriptions.
#
# Supported formats: .jpg, .jpeg, .png
# Recommended minimum: 50+ images across all ripeness stages

# --- Paths ---
# Update IMAGES_DIR to match your Kaggle dataset input name
IMAGES_DIR = '/kaggle/input/my-tomato-images'   # <-- update this
BASE_MODEL  = '/kaggle/input/ripeness-model/ripeness_best.pt'  # <-- update this

STAGING_DIR = '/kaggle/working/staging'
DATASET_DIR = '/kaggle/working/sam3_dataset'
YAML_PATH   = '/kaggle/working/data.yaml'

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

# Gather all image paths
all_images = [
    p for p in Path(IMAGES_DIR).rglob('*')
    if p.suffix.lower() in IMG_EXTENSIONS
]
print(f'Found {len(all_images)} images in {IMAGES_DIR}')
for p in all_images[:5]:
    print(f'  {p.name}')

# Verify base model
base_model = YOLO(BASE_MODEL)
print(f'\nBase model classes: {base_model.names}')

In [ ]:
# Cell 4: SAM3 Auto-Labeling
#
# SAM3 takes plain-English text prompts and finds + segments every matching
# instance in the image. We use one prompt per ripeness class.
#
# The prompts below work well for standard tomatoes. Adjust if your variety
# looks different (e.g. cherry tomatoes: "small unripe cherry tomato").
#
# TEXT_PROMPTS maps prompt index -> unified class ID:
#   index 0 -> class 0 (Unripe)
#   index 1 -> class 1 (Half_Ripe)
#   index 2 -> class 2 (Ripe)

TEXT_PROMPTS = [
    'unripe green tomato',          # -> class 0: Unripe
    'half ripe orange tomato',      # -> class 1: Half_Ripe
    'ripe red tomato',              # -> class 2: Ripe
]
UNIFIED_NAMES = {0: 'Unripe', 1: 'Half_Ripe', 2: 'Ripe'}

# Initialize SAM3 predictor
overrides = dict(
    conf=0.30,          # Confidence threshold. Lower = more detections (more false positives).
                        # Higher = fewer detections (more misses). 0.30 is a good starting point;
                        # lower to 0.20 if SAM3 misses obvious tomatoes.
    task='segment',
    mode='predict',
    model=SAM3_PATH,
    half=True,          # FP16 for faster inference on T4 GPU
)
predictor = SAM3SemanticPredictor(overrides=overrides)

# Output directories
os.makedirs(os.path.join(STAGING_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(STAGING_DIR, 'labels'), exist_ok=True)

labeled_count = 0
skipped_count = 0
class_counts   = Counter()

for img_path in all_images:
    predictor.set_image(str(img_path))

    # Query SAM3 once per class prompt
    # Each call returns all instances matching that text prompt
    all_boxes  = []  # [cls_id, cx, cy, w, h] normalized
    img_w, img_h = None, None

    for cls_id, prompt in enumerate(TEXT_PROMPTS):
        results = predictor(text=[prompt])
        for result in results:
            if result.boxes is None or len(result.boxes) == 0:
                continue
            if img_w is None:
                img_h, img_w = result.orig_shape

            for box in result.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = box[:4]
                # Convert to YOLO normalized format
                cx = ((x1 + x2) / 2) / img_w
                cy = ((y1 + y2) / 2) / img_h
                bw = (x2 - x1) / img_w
                bh = (y2 - y1) / img_h
                # Skip degenerate boxes
                if bw < 0.01 or bh < 0.01:
                    continue
                all_boxes.append((cls_id, cx, cy, bw, bh))
                class_counts[cls_id] += 1

    if not all_boxes:
        # SAM3 found nothing in this image — skip it
        # (common for background/empty images, which is fine)
        skipped_count += 1
        continue

    # Copy image
    dst_img = os.path.join(STAGING_DIR, 'images', img_path.name)
    shutil.copy2(str(img_path), dst_img)

    # Write YOLO label file
    dst_lbl = os.path.join(STAGING_DIR, 'labels', img_path.stem + '.txt')
    with open(dst_lbl, 'w') as f:
        for cls_id, cx, cy, bw, bh in all_boxes:
            f.write(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
    labeled_count += 1

print(f'\nAuto-labeling complete:')
print(f'  Labeled:  {labeled_count} images')
print(f'  Skipped:  {skipped_count} images (no tomatoes found)')
print(f'  Annotations per class:')
for cid in sorted(class_counts):
    print(f'    {UNIFIED_NAMES[cid]}: {class_counts[cid]}')

In [ ]:
# Cell 5: Visualize SAM3 Auto-Labels
#
# Review before training — if SAM3 mislabeled things (e.g. leaves as Unripe),
# adjust TEXT_PROMPTS in Cell 4 and re-run, or delete bad label files manually.

CLASS_COLORS = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}  # Green, Orange, Red

staging_imgs = [
    f for f in os.listdir(os.path.join(STAGING_DIR, 'images'))
    if not f.startswith('.')
]
sample = random.sample(staging_imgs, min(6, len(staging_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_file in enumerate(sample):
    ax = axes[idx]
    img = plt.imread(os.path.join(STAGING_DIR, 'images', img_file))
    h, w = img.shape[:2]
    ax.imshow(img)

    lbl_path = os.path.join(STAGING_DIR, 'labels', os.path.splitext(img_file)[0] + '.txt')
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                color = CLASS_COLORS.get(cls_id, '#fff')
                rect = patches.Rectangle(
                    (x1, y1), bw * w, bh * h,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 4, UNIFIED_NAMES[cls_id],
                        color=color, fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    ax.set_title(img_file[:40], fontsize=9)
    ax.axis('off')

plt.suptitle('SAM3 Auto-Labels — Review Before Training', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/sam3_autolabels.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nIf labels look wrong: adjust TEXT_PROMPTS in Cell 4 and re-run.')

In [ ]:
# Cell 6: Stratified Train/Val/Test Split (70/15/15)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(DATASET_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, split, 'labels'), exist_ok=True)

img_dir = os.path.join(STAGING_DIR, 'images')
lbl_dir = os.path.join(STAGING_DIR, 'labels')

all_stems, all_exts, dominant_classes = [], {}, []

for img_file in sorted(os.listdir(img_dir)):
    if img_file.startswith('.'):
        continue
    stem, ext = os.path.splitext(img_file)
    lbl_path = os.path.join(lbl_dir, stem + '.txt')
    if not os.path.exists(lbl_path):
        continue
    classes = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                classes.append(int(parts[0]))
    if not classes:
        continue
    all_stems.append(stem)
    all_exts[stem] = ext
    dominant_classes.append(Counter(classes).most_common(1)[0][0])

print(f'Total labeled images: {len(all_stems)}')
print(f'Class distribution:   {Counter(dominant_classes)}')

# Need at least 3 samples per class for stratified split
class_counts_check = Counter(dominant_classes)
if min(class_counts_check.values()) < 3:
    print('WARNING: Some classes have fewer than 3 images.')
    print('Falling back to non-stratified split.')
    stratify = None
else:
    stratify = dominant_classes

train_stems, temp_stems, train_labels, temp_labels = train_test_split(
    all_stems, dominant_classes, test_size=0.30, random_state=42, stratify=stratify
)
valid_stems, test_stems, _, _ = train_test_split(
    temp_stems, temp_labels, test_size=0.50, random_state=42,
    stratify=temp_labels if stratify is not None else None,
)

for split_name, stems in [('train', train_stems), ('valid', valid_stems), ('test', test_stems)]:
    for stem in stems:
        ext = all_exts[stem]
        shutil.copy2(os.path.join(img_dir, stem + ext),
                     os.path.join(DATASET_DIR, split_name, 'images', stem + ext))
        shutil.copy2(os.path.join(lbl_dir, stem + '.txt'),
                     os.path.join(DATASET_DIR, split_name, 'labels', stem + '.txt'))

print(f'\nSplit sizes:')
print(f'  train: {len(train_stems)}')
print(f'  valid: {len(valid_stems)}')
print(f'  test:  {len(test_stems)}')

In [ ]:
# Cell 7: Write data.yaml

data_config = {
    'path':  DATASET_DIR,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    3,
    'names': ['Unripe', 'Half_Ripe', 'Ripe'],
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'Classes: {data_config["names"]}')
print(f'\nPer-split annotation counts:')
for split in ['train', 'valid', 'test']:
    counts = Counter()
    for lbl_file in os.listdir(os.path.join(DATASET_DIR, split, 'labels')):
        if not lbl_file.endswith('.txt'):
            continue
        with open(os.path.join(DATASET_DIR, split, 'labels', lbl_file)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    print(f'  {split}: { {data_config["names"][k]: v for k, v in sorted(counts.items())} }')

In [ ]:
# Cell 8: Fine-Tune YOLOv8n-C3TR from ripeness_best.pt
#
# We start from YOUR existing model (ripeness_best.pt) rather than from
# scratch. This means:
#   - The model already knows what tomatoes look like from close-up datasets
#   - We only need to teach it YOUR environment (camera angle, lighting, background)
#   - Much faster convergence (25 epochs vs 100)
#
# Key difference from ripeness_finetune.ipynb:
#   - Labels came from SAM3 on YOUR images → perfect domain match
#   - No more close-up vs whole-plant mismatch
#
# LR is very low (1e-5) because we're nudging an already-good model,
# not relearning from scratch. Too high a LR would destroy the ripeness
# features learned from 3K close-up images.

if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'
    print('WARNING: No GPU — training will be slow')

model = YOLO(BASE_MODEL)

results = model.train(
    data=YAML_PATH,

    # --- Core ---
    epochs=25,            # Short fine-tune. SAM3 labels match our domain exactly
                          # so the model adapts quickly. Increase to 50 if val mAP
                          # is still climbing at epoch 25.
    imgsz=640,
    patience=10,          # Stop early if no improvement for 10 epochs.

    # --- Hardware ---
    device=device,
    workers=4,
    batch=16,             # Smaller batch since dataset is small. Increase to 32
                          # if you have 200+ images.
    amp=True,

    # --- Optimizer ---
    optimizer='AdamW',
    lr0=0.00001,          # 10x lower than ripeness_finetune.ipynb (5e-5).
                          # We're fine-tuning a fine-tuned model — the weights
                          # are already well-adapted to ripeness. We only want
                          # to shift the model's spatial priors (whole-plant
                          # framing) without touching color discrimination.
    warmup_epochs=2,
    cos_lr=True,

    # --- Augmentation ---
    # Conservative color augmentation — ripeness is a color-based task.
    # More geometric augmentation (flips, scale) to handle the variety of
    # angles in whole-plant shots.
    hsv_h=0.01,           # Minimal hue shift — color = class signal
    hsv_s=0.3,
    hsv_v=0.4,
    fliplr=0.5,
    scale=0.3,            # Wider scale range than finetune notebook (0.064)
                          # because whole-plant shots have more zoom variation.
    mosaic=0.5,
    mixup=0.0,            # Disabled — mixing ripeness stages creates impossible colors

    name='ripeness_sam3',
)

In [ ]:
# Cell 9: Validation Results

best_weights = glob.glob('runs/detect/ripeness_sam3*/weights/best.pt')
best_model_path = max(best_weights, key=os.path.getmtime) if best_weights else \
    'runs/detect/ripeness_sam3/weights/best.pt'

print(f'Best model: {best_model_path}')

ft_model = YOLO(best_model_path)
metrics = ft_model.val(data=YAML_PATH, device=device, verbose=True)

print(f'\n{"="*50}')
print(f'  VALIDATION RESULTS')
print(f'{"="*50}')
print(f'  mAP@50:    {metrics.box.map50:.4f}')
print(f'  mAP@50-95: {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall:    {metrics.box.mr:.4f}')
print(f'{"="*50}')

print(f'\nPer-class mAP@50:')
for i, name in enumerate(data_config['names']):
    print(f'  {name}: {metrics.box.maps[i]:.4f}')

In [ ]:
# Cell 10: Test Set Evaluation

test_yaml_path = '/kaggle/working/data_test.yaml'
test_data_config = data_config.copy()
test_data_config['val'] = 'test/images'
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_data_config, f, default_flow_style=False)

test_metrics = ft_model.val(data=test_yaml_path, device=device, verbose=True)

print(f'\n{"="*50}')
print(f'  TEST RESULTS')
print(f'{"="*50}')
print(f'  mAP@50:    {test_metrics.box.map50:.4f}')
print(f'  mAP@50-95: {test_metrics.box.map:.4f}')
print(f'  Precision: {test_metrics.box.mp:.4f}')
print(f'  Recall:    {test_metrics.box.mr:.4f}')
print(f'{"="*50}')

gap = abs(metrics.box.map50 - test_metrics.box.map50)
print(f'\n  Val mAP@50: {metrics.box.map50:.4f}  |  Test mAP@50: {test_metrics.box.map50:.4f}')
print(f'  Gap: {gap:.3f} {"-- generalizes well" if gap <= 0.05 else "-- possible overfit (too few images)"}')

## Inference with SAHI

**Why SAHI fixes the whole-plant detection problem:**

YOLO was trained on 640x640 images. A whole-plant photo at 1920x1080 means each tomato
is only ~30px wide — tiny specks that YOLO struggles with, and large background regions
that can trigger false positives.

SAHI (Slicing Aided Hyper Inference) solves this by:
1. Slicing the full image into overlapping 320x320 tiles
2. Running YOLO on each tile independently (tomatoes now fill more of the frame)
3. Merging all detections back with NMS to remove duplicates

Result: small tomatoes are detected reliably, and background patches (wall, pot) are
less likely to trigger false positives because they're diluted across many tiles.

In [ ]:
# Cell 11: SAHI Inference on Your Test Images
#
# Compare standard YOLO inference vs SAHI sliced inference side-by-side.
# SAHI will typically find more small tomatoes and produce fewer wall/background
# false positives.

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image_as_pil
import PIL.Image as PILImage

# Load model into SAHI wrapper
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=best_model_path,
    confidence_threshold=0.40,  # Higher than standard 0.25 to cut false positives.
                                 # Raise further (0.50) if you still see false positives.
    device=device,
)

# Run on a few test images
test_img_dir = os.path.join(DATASET_DIR, 'test', 'images')
test_imgs = [f for f in os.listdir(test_img_dir) if not f.startswith('.')]
sample_test = random.sample(test_imgs, min(4, len(test_imgs)))

for img_file in sample_test:
    img_path = os.path.join(test_img_dir, img_file)

    # --- Standard YOLO (for comparison) ---
    standard_result = ft_model.predict(
        source=img_path, conf=0.40, save=False, verbose=False
    )[0]
    n_standard = len(standard_result.boxes) if standard_result.boxes else 0

    # --- SAHI Sliced Inference ---
    sahi_result = get_sliced_prediction(
        img_path,
        sahi_model,
        slice_height=320,          # Tile height. For very high-res images (4K),
                                   # increase to 640. For 1080p, 320 works well.
        slice_width=320,           # Tile width.
        overlap_height_ratio=0.2,  # 20% overlap between tiles prevents missed
        overlap_width_ratio=0.2,   # detections at tile boundaries.
        postprocess_match_threshold=0.5,  # IOU threshold for merging duplicate
                                          # detections from overlapping tiles.
    )
    n_sahi = len(sahi_result.object_prediction_list)

    print(f'{img_file}: Standard={n_standard} detections | SAHI={n_sahi} detections')

    # Save SAHI result
    sahi_result.export_visuals(
        export_dir='/kaggle/working/sahi_results/',
        file_name=f'sahi_{img_file}',
    )

# Display results
for f in glob.glob('/kaggle/working/sahi_results/*.png')[:4]:
    display(Image(filename=f, width=700))

In [ ]:
# Cell 12: Export Final Weights

output_weights = '/kaggle/working/ripeness_sam3.pt'
shutil.copy2(best_model_path, output_weights)

print(f'Saved: {output_weights}')
print(f'Size:  {os.path.getsize(output_weights) / (1024**2):.1f} MB')
print(f'\nUsage with standard YOLO inference:')
print(f'  model = YOLO("ripeness_sam3.pt")')
print(f'  model.predict(source="plant.jpg", conf=0.40)')
print(f'\nUsage with SAHI (recommended for whole-plant images):')
print(f'  from sahi import AutoDetectionModel')
print(f'  from sahi.predict import get_sliced_prediction')
print(f'  sahi_model = AutoDetectionModel.from_pretrained(')
print(f'      model_type="ultralytics",')
print(f'      model_path="ripeness_sam3.pt",')
print(f'      confidence_threshold=0.40,')
print(f'  )')
print(f'  result = get_sliced_prediction(')
print(f'      "plant.jpg", sahi_model,')
print(f'      slice_height=320, slice_width=320,')
print(f'      overlap_height_ratio=0.2, overlap_width_ratio=0.2,')
print(f'  )')

# Clean up staging
shutil.rmtree(STAGING_DIR, ignore_errors=True)
print(f'\nDone.')